# Notebook 09 — solvePnP Explained
### 6D Pose Vision Workshop · Part 4: Classical Pose Estimation

**Estimated time:** 55 minutes  
**Dependencies:** `opencv-contrib-python`, `numpy`, `matplotlib`

> **Grounded in:** video note 2 (*OpenCV Python Pose Estimation for Objects*)

---

## Recommended Watch

> Watch this **before** opening the notebook — it shows solvePnP in action with real objects, covering 2D-3D correspondences, rvec, tvec, and drawFrameAxes.

| Title | Link | Duration |
|---|---|---|
| **OpenCV Python Pose Estimation for Objects (Algorithm and Code)** | [▶ Watch](https://www.youtube.com/watch?v=bs81DNsMrnM) | ~20 min |

---

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install opencv-contrib-python numpy matplotlib -q
    print("Running in Google Colab")
else:
    print("Running locally — make sure your venv is activated")

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

print(f"OpenCV: {cv2.__version__}")

def show_images(pairs, figsize_per=(5, 4)):
    n = len(pairs)
    fig, axes = plt.subplots(1, n, figsize=(figsize_per[0]*n, figsize_per[1]))
    if n == 1: axes = [axes]
    for ax, (img, title) in zip(axes, pairs):
        ax.imshow(img[:,:,::-1] if img.ndim==3 else img, cmap=None if img.ndim==3 else 'gray')
        ax.set_title(title, fontsize=9); ax.axis('off')
    plt.tight_layout(); plt.show()

## Learning Objectives

By the end of this notebook you will:

- Understand the **Perspective-n-Point (PnP) problem** conceptually
- Know what **2D-3D correspondences** are and where they come from
- Understand how the **Levenberg-Marquardt optimizer** minimizes reprojection error
- Use `cv2.solvePnP` and interpret its outputs (`rvec`, `tvec`)
- Convert `rvec` → rotation matrix and back with `cv2.Rodrigues`
- Use `cv2.projectPoints` to reproject and validate the pose
- Know the different PnP algorithms and when to use each

---

## 1. The Perspective-n-Point Problem

### The setup

You have:
- A calibrated camera (you know K and dist from Notebook 07)
- An object with known 3D geometry (you know where its key points are in 3D)
- An image of the object (you can detect the 2D positions of those key points)

**Question:** Where is the object relative to the camera? (i.e., what are R and t?)

This is the **Perspective-n-Point (PnP)** problem: find R and t given n 3D-2D point correspondences.

```
Known:                              Want:
  3D world points                     R  (rotation of object)
  + their 2D image locations    →     t  (translation of object)
  + camera intrinsics K
```

### Visual intuition

```
        3D World                     2D Image
                                    ┌──────────┐
    A (0, 0, 0) ────────────────── •a (120, 80)
    B (5, 0, 0) ────────────────── •b (200, 85)
    C (5, 5, 0) ────────────────── •c (200, 160)
    D (0, 5, 0) ────────────────── •d (120, 160)
        ↑                           └──────────┘
    Known 3D                       Detected 2D
    positions                      positions

  → solvePnP finds R, t that makes each A→a, B→b, C→c, D→d
```

### How many points do you need?

- **Minimum 4 non-coplanar points** for a unique solution (some methods need 6)
- More points → more accurate (overdetermined system → least squares)
- For a flat object (all points on Z=0 plane): minimum 6 with some methods

---

## 2. Reprojection Error — What solvePnP Minimizes

### The objective

For a given candidate pose $(R, t)$, we can project each 3D world point $\mathbf{X}_i$ to a predicted 2D pixel $\hat{\mathbf{x}}_i$:

$$\hat{\mathbf{x}}_i = K[R|t]\mathbf{X}_i$$

The **reprojection error** for point $i$ is the distance between predicted and detected:

$$e_i = \|\mathbf{x}_i^{\text{detected}} - \hat{\mathbf{x}}_i^{\text{projected}}\|_2 \quad \text{(pixels)}$$

The total cost to minimize:

$$E(R, t) = \sum_{i=1}^{n} \|\mathbf{x}_i - K[R|t]\mathbf{X}_i\|^2$$

### The Levenberg-Marquardt optimizer

This is a nonlinear least-squares solver — it's what OpenCV uses by default (method `cv2.SOLVEPNP_ITERATIVE`):

```
1. Start with initial pose estimate (from linear method)
2. Compute Jacobian: ∂(reprojection error) / ∂(R, t)
3. Take a step in the direction that reduces error
4. Repeat until convergence (error change < threshold, or max iters)
```

LM is **robust** (handles near-singularities better than Newton's method) and **fast** (converges in ~10-50 iterations for typical poses).

---

In [ ]:
# Build intuition: manually set up and solve a PnP problem

# Ground truth pose (we know this, to verify solvePnP recovers it)
K = np.array([[800, 0, 320],
              [0,   800, 240],
              [0,   0,   1]], dtype=np.float64)
dist = np.zeros(5)  # assume undistorted image

# True pose: object is 1.5m in front, 0.1m right, slight 20° tilt
theta_true = np.radians(20)
R_true = np.array([[np.cos(theta_true), 0, np.sin(theta_true)],
                   [0,                  1, 0                 ],
                   [-np.sin(theta_true),0, np.cos(theta_true)]])
t_true = np.array([[0.1], [0.0], [1.5]])

rvec_true, _ = cv2.Rodrigues(R_true)

# Define 3D object points (corners of a 20cm x 20cm flat marker)
MARKER_SIZE = 0.20  # 20 cm
object_points = np.array([
    [-MARKER_SIZE/2, -MARKER_SIZE/2, 0],
    [ MARKER_SIZE/2, -MARKER_SIZE/2, 0],
    [ MARKER_SIZE/2,  MARKER_SIZE/2, 0],
    [-MARKER_SIZE/2,  MARKER_SIZE/2, 0],
    # Add center for a 5th point
    [0, 0, 0],
], dtype=np.float64)

# Project to get 'detected' image points (add tiny noise to simulate real detection)
image_points_clean, _ = cv2.projectPoints(object_points, rvec_true, t_true, K, dist)
noise = np.random.normal(0, 0.5, image_points_clean.shape)  # 0.5 pixel noise
image_points = image_points_clean + noise
image_points = image_points.astype(np.float32)

print("Ground truth pose:")
print(f"  R (20° Y-rotation):")
print(f"  {R_true.round(4)}")
print(f"  t: {t_true.flatten()}")
print(f"\n  rvec_true: {rvec_true.flatten().round(4)}")

print(f"\n3D object points (5 marker corners + center):")
print(object_points)
print(f"\nProjected 2D image points (with 0.5px noise):")
print(image_points.reshape(-1, 2).round(2))

## 3. cv2.solvePnP — The API

```python
ret, rvec, tvec = cv2.solvePnP(
    objectPoints,   # (N, 3) float64 — 3D world points
    imagePoints,    # (N, 1, 2) or (N, 2) float32 — 2D detected points
    cameraMatrix,   # K — 3x3 float64
    distCoeffs,     # distortion — (5,) or None if undistorted image
    flags=cv2.SOLVEPNP_ITERATIVE  # algorithm to use
)
# Returns:
#   ret   — True if solution found
#   rvec  — (3,1) rotation vector (Rodrigues)
#   tvec  — (3,1) translation vector (meters)
```

### solvePnP methods

| Flag | Algorithm | Min points | Notes |
|---|---|---|---|
| `SOLVEPNP_ITERATIVE` | Levenberg-Marquardt (default) | 4 | Best accuracy, iterative |
| `SOLVEPNP_P3P` | Algebraic (P3P) | 4 (uses only 3) | Fast, no initial estimate needed |
| `SOLVEPNP_AP3P` | Algebraic P3P variant | 4 | More stable than P3P |
| `SOLVEPNP_EPNP` | Efficient PnP | 4 | Good for many points (N>10) |
| `SOLVEPNP_IPPE` | Homography-based | 4 coplanar | Best for **planar objects** (ArUco markers!) |
| `SOLVEPNP_IPPE_SQUARE` | IPPE for square markers | 4 coplanar | **Best for ArUco** — returns 2 solutions |

> For ArUco markers (flat square objects), use `SOLVEPNP_IPPE_SQUARE` — it's specifically designed for this and returns both possible pose solutions.

In [ ]:
# Run solvePnP and compare with ground truth

ret, rvec_est, tvec_est = cv2.solvePnP(
    object_points,
    image_points,
    K,
    dist,
    flags=cv2.SOLVEPNP_ITERATIVE
)

print(f"solvePnP succeeded: {ret}")
print(f"\nEstimated rvec: {rvec_est.flatten().round(4)}")
print(f"True      rvec: {rvec_true.flatten().round(4)}")
print(f"rvec error:     {np.linalg.norm(rvec_est - rvec_true):.6f} rad")

print(f"\nEstimated tvec: {tvec_est.flatten().round(4)} m")
print(f"True      tvec: {t_true.flatten().round(4)} m")
print(f"tvec error:     {np.linalg.norm(tvec_est - t_true)*100:.2f} cm")

# Convert rvec to rotation matrix
R_est, _ = cv2.Rodrigues(rvec_est)
print(f"\nEstimated R:\n{R_est.round(4)}")
print(f"True      R:\n{R_true.round(4)}")

# Compute reprojection error
projected, _ = cv2.projectPoints(object_points, rvec_est, tvec_est, K, dist)
reproj_err = np.mean(np.linalg.norm(
    image_points.reshape(-1, 2) - projected.reshape(-1, 2), axis=1))
print(f"\nReprojection error: {reproj_err:.4f} px")

In [ ]:
# Interpreting rvec and tvec for robotics

print("Interpreting pose output for robotics:")
print("=" * 50)

# tvec: position of the object origin in camera frame
tx, ty, tz = tvec_est.flatten()
distance = np.linalg.norm(tvec_est)  # Euclidean distance

print(f"\ntranslation (tvec):")
print(f"  X (right): {tx:.3f} m  ({tx*100:.1f} cm)")
print(f"  Y (down):  {ty:.3f} m  ({ty*100:.1f} cm)")
print(f"  Z (fwd):   {tz:.3f} m  ({tz*100:.1f} cm)  ← depth from camera")
print(f"  distance:  {distance:.3f} m")

# rvec → rotation angles
# Rodrigues vector norm = rotation angle, direction = rotation axis
angle_rad = np.linalg.norm(rvec_est)
axis = rvec_est.flatten() / angle_rad
print(f"\nrotation (rvec):")
print(f"  angle: {np.degrees(angle_rad):.2f}°")
print(f"  axis:  {axis.round(3)}")

# For planar objects (Z=0), the tvec gives position relative to camera
# The robotics use case:
print(f"\nRobotics interpretation (ArUco at docking station):")
print(f"  Robot camera sees marker at: right={tx:.2f}m, below={ty:.2f}m, depth={tz:.2f}m")
print(f"  Correction needed: move left {-tx:.2f}m, up {-ty:.2f}m to center on marker")

In [ ]:
# cv2.projectPoints — reproject 3D points to 2D for validation and visualization

# Define axis points in 3D (to draw coordinate frame)
axis_length = 0.1  # 10 cm
axis_points_3d = np.array([
    [0, 0, 0],                  # origin
    [axis_length, 0, 0],        # X axis end
    [0, axis_length, 0],        # Y axis end
    [0, 0, -axis_length],       # Z axis end (pointing toward camera = negative Z in object frame)
], dtype=np.float64)

# Project axis points using estimated pose
axis_projected, _ = cv2.projectPoints(axis_points_3d, rvec_est, tvec_est, K, dist)
origin_px = tuple(axis_projected[0].flatten().astype(int))
x_end_px  = tuple(axis_projected[1].flatten().astype(int))
y_end_px  = tuple(axis_projected[2].flatten().astype(int))
z_end_px  = tuple(axis_projected[3].flatten().astype(int))

# Draw everything on a canvas
canvas = np.full((480, 640, 3), 50, dtype=np.uint8)  # dark background

# Draw detected image points (what we observed)
for pt in image_points.reshape(-1, 2):
    cv2.circle(canvas, tuple(pt.astype(int)), 6, (0, 255, 255), -1)

# Draw reprojected points (what solvePnP predicts)
for pt in projected.reshape(-1, 2):
    cv2.circle(canvas, tuple(pt.astype(int)), 4, (0, 255, 0), 2)

# Draw error lines
for obs, proj in zip(image_points.reshape(-1,2), projected.reshape(-1,2)):
    cv2.line(canvas, tuple(obs.astype(int)), tuple(proj.astype(int)), (255, 100, 100), 1)

# Draw coordinate axes
cv2.arrowedLine(canvas, origin_px, x_end_px, (0, 0, 255), 2, tipLength=0.3)  # X = red
cv2.arrowedLine(canvas, origin_px, y_end_px, (0, 255, 0), 2, tipLength=0.3)  # Y = green
cv2.arrowedLine(canvas, origin_px, z_end_px, (255, 0, 0), 2, tipLength=0.3)  # Z = blue
cv2.putText(canvas, 'X', x_end_px, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1)
cv2.putText(canvas, 'Y', y_end_px, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
cv2.putText(canvas, 'Z', z_end_px, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,0), 1)

# Legend
cv2.putText(canvas, 'Cyan: detected', (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,255), 1)
cv2.putText(canvas, 'Green: reprojected (should overlap)', (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
cv2.putText(canvas, 'Red lines: reprojection error', (10, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,100,100), 1)
cv2.putText(canvas, f'Mean error: {reproj_err:.2f} px', (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)

show_images([(canvas, 'Pose estimation result\nCyan=detected, Green=reprojected, Axes=recovered 6D pose')])

## 4. solvePnPRansac — Robust Pose with Outliers

In real scenes, some point matches may be **wrong** (outliers): misdetected corners, incorrect keypoint matches, etc. Standard `solvePnP` is not robust to outliers — a single bad match can corrupt the entire pose.

**RANSAC** (Random Sample Consensus) solves this:

```
1. Randomly sample a minimal subset of points (4)
2. Compute pose from that subset
3. Count how many OTHER points agree with this pose (inliers)
4. Repeat many times
5. Return the pose that had the most inliers
6. Refine on inliers only
```

In [ ]:
# Demonstrate solvePnPRansac robustness

# Add 2 strong outliers to the image points
image_points_with_outliers = image_points.copy()
image_points_with_outliers[1] += np.array([[[80, -60]]])   # big error on point 1
image_points_with_outliers[3] += np.array([[[-50, 70]]])   # big error on point 3

# Standard solvePnP (no RANSAC) — affected by outliers
ret_std, rvec_std, tvec_std = cv2.solvePnP(
    object_points, image_points_with_outliers, K, dist)

# solvePnPRansac — robust to outliers
ret_rsc, rvec_rsc, tvec_rsc, inliers = cv2.solvePnPRansac(
    object_points,
    image_points_with_outliers,
    K, dist,
    reprojectionError=8.0,  # threshold: points within 8px are inliers
    iterationsCount=100,
    confidence=0.99
)

def pose_error(rvec_est, tvec_est, rvec_true, tvec_true):
    r_err = np.linalg.norm(rvec_est - rvec_true)
    t_err = np.linalg.norm(tvec_est - tvec_true) * 100  # cm
    return r_err, t_err

r_std, t_std = pose_error(rvec_std, tvec_std, rvec_true, t_true)
r_rsc, t_rsc = pose_error(rvec_rsc, tvec_rsc, rvec_true, t_true)

print("Pose estimation with 2 outliers out of 5 points:")
print(f"\nStandard solvePnP:")
print(f"  rotation error: {np.degrees(r_std):.2f}°")
print(f"  translation error: {t_std:.2f} cm")

print(f"\nsolvePnPRansac:")
print(f"  rotation error: {np.degrees(r_rsc):.2f}°")
print(f"  translation error: {t_rsc:.2f} cm")
print(f"  inliers detected: {len(inliers) if inliers is not None else 0}/{len(object_points)}")
if inliers is not None:
    print(f"  inlier indices: {inliers.flatten().tolist()}")
    print(f"  outlier indices: {[i for i in range(len(object_points)) if i not in inliers.flatten()]}")

## 5. Where Do Point Correspondences Come From?

The hardest part of PnP in practice is getting reliable 2D-3D correspondences. Different methods for different situations:

| Source of 3D points | Detection method | Use case |
|---|---|---|
| Known object model (CAD) | Feature matching (ORB, SIFT) | Industrial parts, tools |
| ArUco marker corners | `detectMarkers` | Robotics stations, calibration |
| Chessboard corners | `findChessboardCorners` | Camera calibration, demo |
| Deep learning keypoints | Keypoint regression CNN | Any object with learned model |
| Human body joints | Pose estimation model | Human activity recognition |

For robotics with fixed infrastructure (stations, pallets), **ArUco markers** are the most reliable — zero ambiguity, known 3D geometry, fast detection. We cover this in Part 5.

---

## Recap

| Concept | Key point |
|---|---|
| PnP problem | Find R, t given N pairs of (3D world point, 2D image point) + K |
| Minimum points | 4 non-coplanar (some methods); 6 for planar objects |
| What it minimizes | Reprojection error: distance between detected and projected points |
| Algorithm | Levenberg-Marquardt (default); also P3P, EPNP, IPPE |
| Output | `rvec` (Rodrigues rotation), `tvec` (translation in meters) |
| `tvec` meaning | Position of object origin in camera frame: X=right, Y=down, Z=depth |
| RANSAC | Use when some correspondences might be wrong (outlier rejection) |
| Validate | Reproject with `cv2.projectPoints`, compute mean pixel error |

---

**Next:** `10_pose_with_chessboard.ipynb` — put it all together: calibrate, detect corners, run solvePnP, draw a 3D cube on the chessboard.

## Exercises

In [ ]:
# ============================================================
# EXERCISE 1: Effect of noise on pose accuracy
# ============================================================
# Using the same object_points, K, and R_true/t_true:
#   1. Project object_points to get clean image points
#   2. Add Gaussian noise with sigma = 0.5, 1.0, 2.0, 5.0 pixels
#   3. Run solvePnP for each noise level
#   4. Plot: noise level vs rotation error (degrees) and translation error (cm)
#
# What noise level makes pose estimation unreliable (error > 5° or > 2 cm)?

# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# noise_sigmas = [0.0, 0.5, 1.0, 2.0, 5.0, 10.0]
# r_errors, t_errors = [], []
#
# for sigma in noise_sigmas:
#     noisy_pts = image_points_clean + np.random.normal(0, sigma, image_points_clean.shape)
#     noisy_pts = noisy_pts.astype(np.float32)
#     ret_n, rv_n, tv_n = cv2.solvePnP(object_points, noisy_pts, K, dist)
#     r_err = np.degrees(np.linalg.norm(rv_n - rvec_true))
#     t_err = np.linalg.norm(tv_n - t_true) * 100
#     r_errors.append(r_err)
#     t_errors.append(t_err)
#     print(f"sigma={sigma:.1f}px → rot_err={r_err:.2f}°, t_err={t_err:.2f}cm")
#
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# axes[0].plot(noise_sigmas, r_errors, 'bo-')
# axes[0].axhline(5, color='red', linestyle='--', label='5° threshold')
# axes[0].set_xlabel('Noise (pixels)'); axes[0].set_ylabel('Rotation error (°)')
# axes[0].set_title('Rotation error vs noise'); axes[0].legend()
# axes[1].plot(noise_sigmas, t_errors, 'ro-')
# axes[1].axhline(2, color='red', linestyle='--', label='2 cm threshold')
# axes[1].set_xlabel('Noise (pixels)'); axes[1].set_ylabel('Translation error (cm)')
# axes[1].set_title('Translation error vs noise'); axes[1].legend()
# plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# EXERCISE 2: Draw a 3D box using pose
# ============================================================
# Using rvec_est and tvec_est computed above:
#   1. Define a 3D box (20cm x 20cm x 10cm) sitting on the marker surface (Z=0 plane)
#   2. Project the 8 corners using cv2.projectPoints
#   3. Draw the bottom face (green), top face (blue), and vertical edges (white)
#      on a 640x480 dark canvas
#   4. Display the result

# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# W, H, D = 0.10, 0.10, 0.08  # 10x10x8cm box
# box_3d = np.array([
#     [-W/2, -H/2,  0], [W/2, -H/2,  0], [W/2, H/2,  0], [-W/2, H/2,  0],  # bottom
#     [-W/2, -H/2, -D], [W/2, -H/2, -D], [W/2, H/2, -D], [-W/2, H/2, -D], # top
# ], dtype=np.float64)
#
# box_2d, _ = cv2.projectPoints(box_3d, rvec_est, tvec_est, K, dist)
# box_2d = box_2d.reshape(-1, 2).astype(int)
#
# canvas = np.full((480, 640, 3), 40, dtype=np.uint8)
# bottom = [(0,1),(1,2),(2,3),(3,0)]
# top    = [(4,5),(5,6),(6,7),(7,4)]
# verts  = [(0,4),(1,5),(2,6),(3,7)]
# for i,j in bottom: cv2.line(canvas, tuple(box_2d[i]), tuple(box_2d[j]), (0,200,0), 2)
# for i,j in top:    cv2.line(canvas, tuple(box_2d[i]), tuple(box_2d[j]), (200,100,0), 2)
# for i,j in verts:  cv2.line(canvas, tuple(box_2d[i]), tuple(box_2d[j]), (200,200,200), 1)
# show_images([(canvas, '3D box drawn using solvePnP pose')])